# Prompt to upload file

In [3]:
from google.colab import files

#Prompt to upload your ratings CSV file
uploaded = files.upload()

Saving Ratings.csv to Ratings (1).csv


# Confirm Upload

In [5]:
import os

# Verify the uploaded files in the current working directory
uploaded_files = os.listdir()
print("Files uploaded:", uploaded_files)

if "Ratings.csv" in uploaded_files:
  print("SUCCESS: 'Ratings.csv' uploaded!")
else:
  print(
      "WARNING: 'ratings.csv' not found. Please upload or rename your file."
  )

Files uploaded: ['.config', 'Ratings.csv', 'Ratings (1).csv', 'sample_data']
SUCCESS: 'ratings.csv' uploaded!


# 1. IMPORT LIBRARIES, LOAD AND INSPECT DATASET

In [7]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("Ratings.csv")

# 1. First 10 records
print("--- FIRST 10 RECORDS ---")
display(df.head(10))

# 2. Number of rows and columns
print("\n--- DATASET DIMENSIONS ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# 3. Column names
print("\n--- COLUMN NAMES ---")
print(df.columns.tolist())

# 4. Data types
print("\n--- DATA TYPES ---")
print(df.dtypes)

# 5. Missing values
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

# 6. Duplicate records
print("\n--- DUPLICATE RECORDS ---")
print(f"Total duplicate rows: {df.duplicated().sum()}")

--- FIRST 10 RECORDS ---


,user_id,movie_id,movie_title,rating,timestamp
0,U001,M001,Nairobi Nights,3,2026-01-01T08:00:00Z
1,U004,M001,Nairobi Nights,4,2026-01-06T09:00:00Z
2,U007,M001,Nairobi Nights,2,2026-01-11T10:00:00Z
3,U010,M001,Nairobi Nights,2,2026-01-16T11:00:00Z
4,U013,M001,Nairobi Nights,4,2026-01-21T12:00:00Z
5,U016,M001,Nairobi Nights,3,2026-01-26T13:00:00Z
6,U019,M001,Nairobi Nights,3,2026-01-31T14:00:00Z
7,U022,M001,Nairobi Nights,2,2026-02-05T15:00:00Z
8,U025,M001,Nairobi Nights,2,2026-02-10T16:00:00Z
9,U028,M001,Nairobi Nights,4,2026-02-15T17:00:00Z



--- DATASET DIMENSIONS ---
Rows: 509, Columns: 5

--- COLUMN NAMES ---
['user_id', 'movie_id', 'movie_title', 'rating', 'timestamp']

--- DATA TYPES ---
user_id        object
movie_id       object
movie_title    object
rating          int64
timestamp      object
dtype: object

--- MISSING VALUES ---
user_id        2
movie_id       1
movie_title    1
rating         0
timestamp      0
dtype: int64

--- DUPLICATE RECORDS ---
Total duplicate rows: 3


# 2. Data Cleaning

In [11]:
#2.1 Check duplicates
initial_duplicates = df.duplicated().sum()
print(f"2.1 Number of duplicate rows found: {initial_duplicates}")

#2.1.1 Remove duplicates
df_clean = df.drop_duplicates().copy()
print(
    f"2.1.1 Removed {initial_duplicates} duplicate rows. Remaining rows:"
    f" {len(df_clean)}"
)

#2.2 Check missing user/movie IDs
missing_users = df_clean["user_id"].isnull().sum()
missing_movies = df_clean["movie_id"].isnull().sum()
print(
    f"2.2 Missing values - user_id: {missing_users}, movie_id:"
    f" {missing_movies}"
)

#2.2.1 Remove missing IDs
df_clean = df_clean.dropna(subset=["user_id", "movie_id"]).copy()
print(
    f"2.2.1 Removed rows with missing IDs. Remaining rows: {len(df_clean)}"
)

#2.3 Check the rating scale
min_rating = df_clean["rating"].min()
max_rating = df_clean["rating"].max()
unique_ratings = sorted(df_clean["rating"].unique())

print("\n2.3 Rating Scale Inspection:")
print(f"    Minimum Rating: {min_rating}")
print(f"    Maximum Rating: {max_rating}")
print(f"    Unique Rating Values Found: {unique_ratings}")

#Identify invalid ratings outside standard 1–5 scale
invalid_mask = ~df_clean["rating"].isin([1, 2, 3, 4, 5])
invalid_count = invalid_mask.sum()
print(f"    Invalid ratings outside 1–5 scale found: {invalid_count}")

#Remove invalid ratings
df_clean = df_clean[df_clean["rating"].isin([1, 2, 3, 4, 5])].copy()

#2.4 Final cleaning check
print("\n2.4 Final Cleaning Verification:")
print(
    f"    Final Dataset Shape: {df_clean.shape[0]} rows,"
    f" {df_clean.shape[1]} columns"
)
print(f"    Remaining Missing Values: {df_clean.isnull().sum().sum()}")
print(f"    Remaining Duplicates: {df_clean.duplicated().sum()}")

2.1 Number of duplicate rows found: 3
2.1.1 Removed 3 duplicate rows. Remaining rows: 506
2.2 Missing values - user_id: 2, movie_id: 1
2.2.1 Removed rows with missing IDs. Remaining rows: 503

2.3 Rating Scale Inspection:
    Minimum Rating: -1
    Maximum Rating: 6
    Unique Rating Values Found: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
    Invalid ratings outside 1–5 scale found: 3

2.4 Final Cleaning Verification:
    Final Dataset Shape: 500 rows, 5 columns
    Remaining Missing Values: 0
    Remaining Duplicates: 0


#3. EXPLORE THE INTERACTION DATA

In [12]:
#3.1 Total number of ratings
total_ratings = len(df_clean)

#3.2 Number of unique users
num_users = df_clean["user_id"].nunique()

#3.3 Number of unique movies
num_movies = df_clean["movie_id"].nunique()

#3.4 Most active user
user_counts = df_clean["user_id"].value_counts()
most_active_user_id = user_counts.index[0]
most_active_user_ratings = user_counts.iloc[0]

#3.5 Most-rated movie
movie_counts = df_clean["movie_id"].value_counts()
most_rated_movie_id = movie_counts.index[0]
most_rated_movie_ratings = movie_counts.iloc[0]

#3.6 Average rating (overall)
overall_avg_rating = round(df_clean["rating"].mean(), 2)

#Print individual results
print(f"3.1 Total Number of Ratings: {total_ratings}")
print(f"3.2 Number of Unique Users: {num_users}")
print(f"3.3 Number of Unique Movies: {num_movies}")
print(
    f"3.4 Most Active User: {most_active_user_id} ({most_active_user_ratings}"
    " ratings)"
)
print(
    f"3.5 Most-Rated Movie: {most_rated_movie_id} ({most_rated_movie_ratings}"
    " ratings)"
)
print(f"3.6 Overall Average Rating: {overall_avg_rating}")

#3.7 Summary Table
summary_data = {
    "Metric": [
        "Total Ratings",
        "Unique Users",
        "Unique Movies",
        "Most Active User ID",
        "Max Ratings by Single User",
        "Most-Rated Movie ID",
        "Max Ratings for Single Movie",
        "Overall Average Rating",
    ],
    "Value": [
        total_ratings,
        num_users,
        num_movies,
        most_active_user_id,
        most_active_user_ratings,
        most_rated_movie_id,
        most_rated_movie_ratings,
        overall_avg_rating,
    ],
}

summary_df = pd.DataFrame(summary_data)
print("\n--- 3.7 INTERACTION SUMMARY TABLE ---")
display(summary_df)

3.1 Total Number of Ratings: 500
3.2 Number of Unique Users: 50
3.3 Number of Unique Movies: 24
3.4 Most Active User: U010 (17 ratings)
3.5 Most-Rated Movie: M001 (37 ratings)
3.6 Overall Average Rating: 3.45

--- 3.7 INTERACTION SUMMARY TABLE ---


,Metric,Value
0,Total Ratings,500
1,Unique Users,50
2,Unique Movies,24
3,Most Active User ID,U010
4,Max Ratings by Single User,17
5,Most-Rated Movie ID,M001
6,Max Ratings for Single Movie,37
7,Overall Average Rating,3.45


# 4. User-Item Interaction Matrix

In [13]:
#Create the user-item interaction matrix
interaction_matrix = df_clean.pivot_table(
    index="user_id", columns="movie_id", values="rating"
)

#Display dimensions
print("--- MATRIX DIMENSIONS ---")
print(
    f"Rows (Users): {interaction_matrix.shape[0]}, Columns (Movies):"
    f" {interaction_matrix.shape[1]}"
)

#Display a small sample (5 users x 10 movies)
print("\n--- SAMPLE INTERACTION MATRIX (5 Users x 10 Movies) ---")
display(interaction_matrix.iloc[:5, :10])

--- MATRIX DIMENSIONS ---
Rows (Users): 50, Columns (Movies): 24

--- SAMPLE INTERACTION MATRIX (5 Users x 10 Movies) ---


movie_id,M001,M002,M003,M004,M005,M006,M007,M008,M009,M010
user_id,,,,,,,,,,
U001,3.0,2.0,NaN,NaN,4.0,4.0,NaN,NaN,NaN,NaN
U002,3.0,NaN,NaN,3.5,NaN,NaN,3.0,NaN,NaN,NaN
U003,NaN,NaN,NaN,NaN,4.0,NaN,NaN,4.0,4.0,NaN
U004,4.0,NaN,NaN,NaN,3.0,3.0,NaN,NaN,NaN,NaN
U005,2.0,4.0,NaN,3.0,NaN,NaN,5.0,NaN,NaN,NaN


# 5. Dataset Sparsity

### Sparsity Formula:
$$\text{Sparsity} = 1 - \left( \frac{\text{Observed Ratings}}{\text{Number of Users} \times \text{Number of Movies}} \right)$$

In [14]:
#1. Number of users and movies
n_users = df_clean["user_id"].nunique()
n_movies = df_clean["movie_id"].nunique()

#2. Total possible interactions
possible_interactions = n_users * n_movies

#3. Observed ratings
observed_ratings = len(df_clean)

#4. Sparsity calculation
sparsity = 1 - (observed_ratings / possible_interactions)
sparsity_pct = sparsity * 100

#Display calculation results
print(f"Number of Users: {n_users}")
print(f"Number of Movies: {n_movies}")
print(f"Possible User-Movie Interactions: {possible_interactions}")
print(f"Observed Ratings: {observed_ratings}")
print(f"Calculated Sparsity: {sparsity:.4f}")
print(f"Sparsity Percentage: {sparsity_pct:.2f}%")

Number of Users: 50
Number of Movies: 24
Possible User-Movie Interactions: 1200
Observed Ratings: 500
Calculated Sparsity: 0.5833
Sparsity Percentage: 58.33%


# 6. Popularity-Based Recommender

In [15]:
#1. Group by movie_id and calculate mean rating and rating count
movie_stats = (
    df_clean.groupby(["movie_id", "movie_title"])
    .agg(
        average_rating=("rating", "mean"), rating_count=("rating", "count")
    )
    .reset_index()
)

#Round average rating to 2 decimal places
movie_stats["average_rating"] = movie_stats["average_rating"].round(2)

#2. Filter: Keep movies with at least 10 ratings
filtered_movies = movie_stats[movie_stats["rating_count"] >= 10].copy()

#3. Sort primarily by average_rating, secondarily by rating_count (both descending)
top_10_recommendations = filtered_movies.sort_values(
    by=["average_rating", "rating_count"], ascending=[False, False]
).head(10)

#Display the final Top 10 DataFrame
print("--- TOP 10 RECOMMENDATIONS ---")
display(
    top_10_recommendations[
        ["movie_id", "movie_title", "average_rating", "rating_count"]
    ]
)

--- TOP 10 RECOMMENDATIONS ---


,movie_id,movie_title,average_rating,rating_count
20,M021,Market Day,4.47,15
12,M013,Midnight in Mombasa,4.35,17
13,M014,Code Breakers,4.25,16
18,M019,Future Africa,4.13,15
19,M020,Home Again,4.13,15
6,M007,Hidden Lake,4.12,16
5,M006,City of Tomorrow,3.91,34
4,M005,Mountain Echoes,3.86,37
11,M012,The Final Goal,3.76,17
3,M004,Coastal Dreams,3.67,27


# 7. Export Top 10 Recommendations

In [16]:
from google.colab import files

# Save the top 10 recommendations to CSV without the dataframe index
export_cols = ["movie_id", "movie_title", "average_rating", "rating_count"]
top_10_recommendations[export_cols].to_csv(
    "top_10_recommendations.csv", index=False
)

print("SUCCESS: 'top_10_recommendations.csv' created.")

# Download the file to your computer
files.download("top_10_recommendations.csv")

SUCCESS: 'top_10_recommendations.csv' created.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 8. Interpretation of Results

### 8.1 Why Recommended Movies Appear at the Top
The recommended movies placed at the top of our ranking based on three specific criteria:
1. **High Average Rating:** Movies like *Market Day* (`M021`, avg rating: 4.47) and *Midnight in Mombasa* (`M013`, avg rating: 4.35) achieved the highest overall user satisfaction scores.
2. **Minimum Interaction Threshold:** A mandatory filter of `rating_count >= 10` was enforced. This removed niche items with small sample sizes (e.g., a movie with 1 single 5-star rating) to ensure high statistical confidence.
3. **Secondary Sorting by Popularity:** Where average ratings were identical (e.g., `M019` and `M020` both averaging 4.13), the tie was broken by selecting the movie with higher interaction volume (`rating_count`).

### 8.2 Advantage of Popularity-Based Recommenders
* **Cold-Start Efficiency & Simplicity:** Popularity-based recommenders do not require user interaction history to make recommendations. They provide immediate, high-quality recommendations for new users (solving the user cold-start problem) and are computationally lightweight to compute and deploy.

### 8.3 Limitations of Popularity-Based Recommenders
1. **Lack of Personalization:** Every user receives the exact same set of recommendations regardless of their unique personal preferences, taste profiles, or past watching history.
2. **Popularity Bias (The Long-Tail Problem):** The algorithm heavily favors widely seen, mainstream titles while ignoring niche, less-popular items ("long-tail" movies) that specific users might enjoy much more.

# 9. Conclusion

This project successfully evaluated movie interaction data and established a popularity-based baseline recommender model:

1. **Data Cleaning:** The initial raw dataset contained 509 entries. After dropping 3 duplicate rows, removing 3 entries with missing IDs, and filtering out 3 invalid ratings outside the standard 1–5 scale, a clean dataset of 500 valid ratings was established.
2. **Interaction Statistics:** The catalog comprises 50 unique users and 24 unique movies, with an overall average rating of 3.45 across all interactions.
3. **Sparsity Analysis:** The user-item interaction matrix exhibits a sparsity of 58.33% (500 observed ratings out of 1,200 potential user-movie combinations), illustrating the classic sparse matrix challenge inherent to recommendation systems.
4. **Popularity Recommender:** By enforcing a threshold of `rating_count >= 10` and ranking by average rating, movie `M021` (*Market Day*) emerged as the top recommendation with an average rating of 4.47 across 15 ratings.
5. **Model Evaluation:** While popular items serve as an effective cold-start baseline, future implementations must integrate personalized collaborative filtering or content-based methods to address popularity bias and tailor recommendations to individual user preferences.

# 10.README.md

## Project Description
This repository contains a step-by-step implementation of a popularity-based movie recommendation baseline system built using Python and Pandas. The project demonstrates the full machine learning pipeline for recommendation systems, including data inspection, cleaning, user-item interaction matrix generation, sparsity calculation, baseline recommendation modeling, and output export.

## Objectives
- Perform data inspection and cleaning on raw rating interaction logs.
- Construct a User-Item Interaction Matrix to structure explicit rating data.
- Measure and explain dataset sparsity.
- Build a Non-Personalized Popularity-Based Recommender System using average ratings and interaction frequency thresholds.
- Export top recommendation baselines to CSV for downstream deployment.

## Dataset
The dataset `ratings.csv` represents user rating logs with the following columns:
- **`user_id`**: Unique string identifier for each user (e.g., `U001`).
- **`movie_id`**: Unique string identifier for each movie (e.g., `M001`).
- **`movie_title`**: Descriptive title of the movie (e.g., *Nairobi Nights*).
- **`rating`**: Explicit feedback score on a 1–5 scale.
- **`timestamp`**: ISO-8601 formatted timestamp recording when the rating occurred.

## Methodology
1. **Data Inspection**: Examined initial dataset structure (509 rows, 5 columns), column data types, missing values, and row duplicates.
2. **Data Cleaning**:
   - Removed 3 exact duplicate rows.
   - Removed 3 records containing missing `user_id` or `movie_id` fields.
   - Identified and filtered out 3 invalid ratings outside the standard 1–5 scale (`-1`, `0`, `6`).
   - Resulted in a clean dataset of 500 valid ratings.
3. **Interaction Analysis**: Calculated interaction metrics across 50 unique users and 24 unique movies.
4. **User-Item Matrix**: Formed a $50 \\times 24$ interaction matrix where values correspond to explicit user ratings.
5. **Sparsity Calculation**: Calculated sparsity using the formula:
   $$\\text{Sparsity} = 1 - \\left( \\frac{\\text{Observed Ratings}}{\\text{Users} \\times \\text{Movies}} \\right)$$
   Achieved a matrix sparsity of **58.33%** (500 observed ratings / 1,200 total possible interactions).
6. **Popularity Recommendation**: Grouped data by movie, calculated average ratings and rating counts, applied a threshold of `rating_count >= 10`, and sorted by average rating (descending) and rating count (descending).

## Key Findings
- **Cleaned Dataset Size**: 500 records
- **Unique Users**: 50
- **Unique Movies**: 24
- **Overall Average Rating**: 3.45 / 5.00
- **Most Active User**: `U010` (17 ratings)
- **Most Rated Movie**: `M001` / *Nairobi Nights* (37 ratings)
- **Top Recommended Movie**: `M021` / *Market Day* (Average Rating: 4.47, Rating Count: 15)

## Limitations
1. **Lack of Personalization**: The model delivers identical recommendations to all users regardless of individual preferences or historical viewing habits.
2. **Popularity Bias**: Less-popular or niche items ("long-tail" catalog items) are systematically overlooked even if they closely align with specific user interests.

## Repository Structure